# DE7 - Hessen

In [1]:
import os

import zipfile
import pandas as pd
from tqdm import tqdm
import pyproj
from camelsp import get_metadata

from camels_de1h import get_input_path, get_nuts_id_from_provider_id, Station1h

## Extract data

>Die Abflussdaten sind bis einschließlich 31.12.2019 geprüft. Für die Auswertungen sollten jeweils nur die relevanten Informationen zu Datum und Uhrzeit sowie „Wert [Kubikmeter pro Sekunde]“ herangezogen werden. Aus technischen Gründen war es leider nicht möglich, beim Export der Abflusswerte lediglich die drei signifikanten Stellen auszugeben.

In [ ]:
input_path = get_input_path("DE7")

# extract zip
if not (input_path / "Q").exists():
    os.makedirs(input_path / "Q")
    
    # extract the zip file
    with zipfile.ZipFile(input_path / "KI-HopE-De_Hessen_Q15.zip", "r") as zip:
        zip.extractall(input_path / "Q")

    # move the metadata excel file to the input path
    metadata_path = input_path / "Q" / "KI-HopE-De_Hessen_Pegelstammdaten.xlsx"
    metadata_path.rename(input_path / "KI-HopE-De_Hessen_Pegelstammdaten.xlsx")
        
    print("Data extracted.")
else:
    print("Data already extracted.")

Data already extracted.


## Parse data

**Timezone: UTC+1 (MEZ)**

Data is in 15 minute resolution.

In [3]:
raw_meta_all = pd.read_excel(input_path / "KI-HopE-De_Hessen_Pegelstammdaten.xlsx")
raw_meta_all["STAT_NUM"] = raw_meta_all["STAT_NUM"].astype(str)

ids = raw_meta_all["STAT_NUM"].values

raw_meta_all.head(10)

,STAT_NAME,STAT_NUM,KURZ_NAME,RW,HW,RW_UTM,HW_UTM,BREITE,LÄNGE,BETREIBER,...,HOEHE_GUELTIG,HOEHENSYSTEM,BEZUG_TALSHRB,GEW_KENN,GEW_NAME,GEB_KENN,FLUSSGEB,EZG,UFER,KM
0,Adelshausen,42780500,Adelshausen,3539293,5664118,539203.32,5662292.03,51.110769,9.560030,RP Kassel,...,2009-01-01,Höhe DHHN92 ( HS 160) / 160,NaN,4278,Pfieffe,427891000,Weser,116.16,L,1.00
1,Alsfeld,42880458,Alsfeld,3520196,5624741,520113.63,5622930.81,50.757801,9.285161,RP Kassel,...,2015-05-21,Höhe DHHN92 ( HS 160) / 160,NaN,4288,Schwalm,428813000,Weser,131.41,R,74.20
2,Angenrod,42881009,Angenrod,3514647,5625240,514566.84,5623429.65,50.762453,9.206542,WV Schwalm,...,2010-08-25,Höhe DHHN92 ( HS 160) / 160,NaN,42882,Antreff,428825300,Weser,58.74,L,18.00
3,Aßlar,25842500,Aßlar,3462100,5605350,462040.59,5603547.93,50.582606,8.463828,RP Gießen,...,1979-06-07,vorläufige Höhe (Höhenst 130) / 30,NaN,2584,Dill,258497100,Rhein,693.63,R,5.00
4,Auhammer,42810204,Auhammer,3473712,5655658,473648.32,5653835.87,51.035464,8.624170,RP Kassel,...,2009-01-01,Höhe DHHN92 ( HS 160) / 160,NaN,428,Eder,428175100,Weser,488.94,L,110.00
5,Bad_Hersfeld1,42710050,Bad Hersfeld1,3550760,5636750,550665.57,5634934.81,50.863879,9.719944,RPU Bad Hersfeld,...,1967-11-01,vorläufige Höhe (Höhenst 130) / 30,NaN,42,Fulda,427110000,Weser,2120.29,L,119.80
6,Bad_Hersfeld2,42590301,Bad Hersfeld2,3549350,5637890,549256.14,5636074.37,50.874248,9.700071,RP Kassel,...,1962-07-01,vorläufige Höhe (Höhenst 130) / 30,NaN,42596,Geis,425969300,Weser,74.90,L,1.85
7,Bad_Salzschlirf,42430156,Bad Salzschlirf,3535810,5609830,535721.32,5608025.62,50.623014,9.504991,RP Kassel,...,2006-08-29,vorläufige Höhe (Höhenst 130) / 30,NaN,424,Altefeldbach,424370000,Weser,135.10,R,0.53
8,Bad_Soden,24781909,Bad Soden,3526130,5572680,526044.91,5570890.45,50.289564,9.365616,RPU Frankfurt,...,1958-11-01,vorläufige Höhe (Höhenst 130) / 30,NaN,24782,Salz,247829100,Rhein,89.10,R,1.70
9,Bad_Vilbel,24870055,Bad Vilbel,3481580,5560710,481512.55,5558925.52,50.182241,8.741057,RPU Frankfurt,...,1998-07-01,vorläufige Höhe (Höhenst 130) / 30,NaN,248,Nidda,248730000,Rhein,1617.77,R,22.00


In [4]:
len(list((input_path / "Q").iterdir()))

106

106 stations in metadata, 106 data files. Seems to be okay.

In [5]:
camelsp_meta = get_metadata()
camelsp_meta[camelsp_meta["provider_id"].isin(ids)]

,camels_id,provider_id,camels_path,nuts_lvl2,federal_state,gauge_name,waterbody_name,gauge_elevation,area,x,...,q_extent_years,w_extent_years,q_w_pearson,q_w_spearman,merit_hydro_available,q_more_than_10_years,merit_area_greater_5_smaller_15000,merit_completely_within_germany,merit_difference_to_reported_area_smaller_10_percent,merit_difference_to_reported_area_smaller_20_percent
1658,DE710000,42780500,./DE7/DE710000/DE710000_data.csv,DE7,Hessen,Adelshausen,Pfieffe,171.238,116.16,4.290189e+06,...,41.191781,37.189041,0.757304,0.746964,True,True,True,True,True,True
1659,DE710010,42880458,./DE7/DE710010/DE710010_data.csv,DE7,Hessen,Alsfeld,Schwalm,237.682,131.41,4.270558e+06,...,54.202740,36.189041,0.933412,0.977529,True,True,True,True,True,True
1660,DE710020,42881009,./DE7/DE710020/DE710020_data.csv,DE7,Hessen,Angenrod,Antreff,280.930,58.74,4.265016e+06,...,45.194521,36.189041,0.956833,0.956523,True,True,True,True,True,True
1661,DE710030,25842500,./DE7/DE710030/DE710030_data.csv,DE7,Hessen,Aßlar,Dill,153.030,693.63,4.212203e+06,...,59.205479,37.189041,0.958404,0.997810,True,True,True,True,True,True
1662,DE710040,42810204,./DE7/DE710040/DE710040_data.csv,DE7,Hessen,Auhammer,Eder,298.216,488.94,4.224498e+06,...,62.208219,36.189041,0.938294,0.991793,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1750,DE710920,24810600,./DE7/DE710920/DE710920_data.csv,DE7,Hessen,Unter-Schmitten,Nidda,132.300,124.00,4.251688e+06,...,55.202740,34.189041,0.898677,0.750944,True,True,True,True,True,True
1751,DE710930,42882806,./DE7/DE710930/DE710930_data.csv,DE7,Hessen,Uttershausen,Schwalm,164.411,984.98,4.273917e+06,...,64.208219,36.189041,0.916939,0.773296,True,True,True,True,True,True
1752,DE710940,24782800,./DE7/DE710940/DE710940_data.csv,DE7,Hessen,Weilers,Bracht,140.790,111.90,4.271860e+06,...,50.200000,34.189041,0.945668,0.941708,True,True,True,True,True,True
1753,DE710950,24861407,./DE7/DE710950/DE710950_data.csv,DE7,Hessen,Windecken,Nidder,112.620,392.60,4.240648e+06,...,66.210959,34.189041,0.904685,0.741318,True,True,True,True,True,True


In [6]:
def parse_station_data(gauge_name_file: str, aggregate_hourly: bool) -> pd.DataFrame:
    # read q data for the station
    data = pd.read_csv(input_path / "Q" / f"{gauge_name_file}-Q15.csv", encoding="latin1", sep=";", usecols=["Datum/Uhrzeit", "Wert [Kubikmeter pro Sekunde]"],
                    parse_dates=["Datum/Uhrzeit"], date_format="%d.%m.%Y %H:%M:%S", index_col=False, na_values=["---"])
    data = data.rename(columns={"Datum/Uhrzeit": "date", "Wert [Kubikmeter pro Sekunde]": "discharge_vol_obs"})

    # transform date time to UTC+0 (data is in UTC+1)
    data["date"] = data["date"] - pd.Timedelta(hours=1)
    data["date"] = data["date"].dt.tz_localize("UTC")

    # add empty water level column
    data["water_level_obs"] = None

    if aggregate_hourly:
        data = data.set_index("date").resample("h", label="right", closed="right").mean().reset_index()

    return data

parse_station_data("Weilers", aggregate_hourly=True)

,date,discharge_vol_obs,water_level_obs
0,1987-10-31 23:00:00+00:00,0.957454,NaN
1,1987-11-01 00:00:00+00:00,0.955175,NaN
2,1987-11-01 01:00:00+00:00,0.951487,NaN
3,1987-11-01 02:00:00+00:00,0.947751,NaN
4,1987-11-01 03:00:00+00:00,0.943971,NaN
...,...,...,...
327236,2025-02-28 19:00:00+00:00,1.777915,NaN
327237,2025-02-28 20:00:00+00:00,1.752122,NaN
327238,2025-02-28 21:00:00+00:00,1.713906,NaN
327239,2025-02-28 22:00:00+00:00,1.701167,NaN


## Run for all stations

I checked consistency between raw metadata and cameslp metadata, there are just some small differences for some gauge names (e.g. `{'raw': 'Dillenburg2', 'camelsp': 'Dillenburg 2'}`), so we can just use the camelsp metadata, if available.  
For the 10 stations that are not in the camelsp metadata, we use the raw metadata.

In [7]:
# get the metadata from CAMELS-DE daily
camelsp_meta_all = get_metadata()

for id in tqdm(ids):
    # get the gauge_id from the nuts_mapping, add it if it does not exist
    gauge_id = get_nuts_id_from_provider_id(id, "DE7", add_missing=True)

    # get the gauge name to read the data
    gauge_name_file = raw_meta_all[raw_meta_all["STAT_NUM"] == id]["STAT_NAME"].values[0]

    # parse data for the station
    data = parse_station_data(gauge_name_file, aggregate_hourly=True)

    # get the metadata of the station
    camelsp_meta = camelsp_meta_all[camelsp_meta_all["camels_id"] == gauge_id]
    raw_meta = raw_meta_all[raw_meta_all["STAT_NUM"] == id]

    if not camelsp_meta.empty:
        # set metadata values from camelsp metadata
        station_meta = {
            "gauge_id": camelsp_meta["camels_id"].values[0],
            "provider_id": camelsp_meta["provider_id"].values[0],
            "gauge_name": camelsp_meta["gauge_name"].values[0],
            "waterbody_name": camelsp_meta["waterbody_name"].values[0],
            "federal_state": camelsp_meta["federal_state"].values[0],
            "gauge_lon": camelsp_meta["lon"].values[0],
            "gauge_lat": camelsp_meta["lat"].values[0],
            "gauge_easting": camelsp_meta["x"].values[0],
            "gauge_northing": camelsp_meta["y"].values[0],
            "gauge_elev_metadata": camelsp_meta["gauge_elevation"].values[0],
            "area_metadata": camelsp_meta["area"].values[0],
            "part_of_camelsp": True,
        }
    elif not raw_meta.empty:
        # parse the location
        easting, northing = raw_meta["RW_UTM"].values[0], raw_meta["HW_UTM"].values[0]

        # from EPSG:25832 to EPSG:4326
        transformer = pyproj.Transformer.from_crs("epsg:25832", "epsg:4326", always_xy=True)
        lon, lat = transformer.transform(easting, northing)

        # from EPSG:25832 to EPSG:3035
        transformer = pyproj.Transformer.from_crs("epsg:25832", "epsg:3035", always_xy=True)
        easting, northing = transformer.transform(easting, northing)

        # if the station is not in the camelsp metadata, use the raw metadata
        station_meta = {
            "gauge_id": gauge_id,
            "provider_id": id,
            "gauge_name": raw_meta["KURZ_NAME"].values[0],
            "waterbody_name": raw_meta["GEW_NAME"].values[0],
            "federal_state": "Hessen",
            "gauge_lon": lon,
            "gauge_lat": lat,
            "gauge_easting": easting,
            "gauge_northing": northing,
            "gauge_elev_metadata": raw_meta["HOEHE_PNP"].values[0],
            "area_metadata": raw_meta["EZG"].values[0],
            "part_of_camelsp": False,
        }
    else:
        raise ValueError(f"Station {id} has no metadata.")
    
    # initialize the station
    station = Station1h(gauge_id)

    # save time series data
    station.save_data(data)

    # save metadata
    station.save_metadata(**station_meta)
    
    # save raw metadata
    station.save_raw_metadata(raw_meta)
    

100%|██████████| 106/106 [45:59<00:00, 26.03s/it]
